In [5]:
from sklearn.model_selection import RandomizedSearchCV

print("Iniciando exploración amplia con RandomizedSearchCV")
random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter=10, 
    cv=5, 
    random_state=42, 
    n_jobs=-1,
    verbose=0 
)

# Entrenamos el modelo
random_search.fit(X, y)


print("\n Búsqueda finalizada con éxito.")
print("Mejores hiperparámetros sugeridos por RandomizedSearch:")
print("-" * 55)

for parametro, valor in random_search.best_params_.items():
    nombre_parametro = parametro.replace('regressor__', '')
    print(f" • {nombre_parametro:<25} : {valor}")

Iniciando exploración amplia con RandomizedSearchCV

 Búsqueda finalizada con éxito.
Mejores hiperparámetros sugeridos por RandomizedSearch:
-------------------------------------------------------
 • n_estimators              : 200
 • min_samples_split         : 10
 • max_depth                 : 10


Siguiendo el marco metodológico avanzado, se utiliza primero **RandomizedSearchCV** para explorar el espacio de hiperparámetros de forma eficiente, identificando las regiones de interés. Posteriormente, se aplica un **GridSearchCV** para el ajuste fino (explotación). Esta estrategia híbrida demuestra un uso eficiente de los recursos computacionales y asegura encontrar la configuración óptima del modelo.

In [6]:
import sys
import pandas as pd
from sklearn.model_selection import GridSearchCV
import joblib
import os

sys.path.append('..')
from src.data_preprocessing import load_gym_data
from src.model_training import get_regression_pipeline
from src.model_evaluation import evaluate_model_cv

# Cargar Datos
df = load_gym_data('../data/gym_members_exercise_tracking.csv')

X = df.drop(columns=['Fat_Percentage'])
y = df['Fat_Percentage']

# Traemos el pipeline de regresión (Random Forest)
rf_pipeline = get_regression_pipeline(model_type='random_forest')

# 1. Definimos la grilla de hiperparámetros
# NOTA: Al usar un Pipeline, debemos poner el prefijo del paso ('regressor__')
param_grid = {
    'regressor__n_estimators': [100, 200, 300], # Número de árboles
    'regressor__max_depth': [None, 10, 20],     # Profundidad para evitar sobreajuste
    'regressor__min_samples_split': [2, 5, 10]  # Mínimo de muestras para dividir un nodo
}

print("Iniciando GridSearchCV... Esto puede tomar unos minutos dependiendo de los núcleos de tu CPU.")

# 2. Configurar GridSearchCV
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5, # 5 Folds para validación cruzada
    scoring='neg_mean_squared_error',
    n_jobs=-1, # Usar todos los procesadores disponibles
    verbose=1
)

# 3. Ejecutar la búsqueda
grid_search.fit(X, y)

# 4. Resultados
print(f"\nMejores hiperparámetros encontrados:\n{grid_search.best_params_}")
print(f"Mejor score (Neg MSE): {grid_search.best_score_:.4f}")

# 5. Guardar el modelo optimizado como la versión final para producción
best_model = grid_search.best_estimator_
MODELS_DIR = '../models/trained_models/'
joblib.dump(best_model, os.path.join(MODELS_DIR, 'optimized_rf_regressor.pkl'))
print("\nModelo optimizado guardado exitosamente en models/trained_models/optimized_rf_regressor.pkl")

Iniciando GridSearchCV... Esto puede tomar unos minutos dependiendo de los núcleos de tu CPU.
Fitting 5 folds for each of 27 candidates, totalling 135 fits

Mejores hiperparámetros encontrados:
{'regressor__max_depth': 10, 'regressor__min_samples_split': 10, 'regressor__n_estimators': 200}
Mejor score (Neg MSE): -7.9398

Modelo optimizado guardado exitosamente en models/trained_models/optimized_rf_regressor.pkl
